# Agent 04 - Refresh Ping Range Master (Polygon)

## Funcion

Construir/actualizar la tabla maestra `ping_range_master` desde Polygon para saber, por ticker:
- `has_data`
- `first_day`
- `last_day`
- `updated_at_utc`

## Como debemos hacerlo

1. Cargar universo de tickers (o delta) desde parquet local.
2. Hacer ping a `/v2/aggs/ticker/{ticker}/range/1/day/{start}/{end}` en `asc` y `desc` (`limit=1`).
3. Guardar resultado en parquet/csv maestra.
4. Ejecutar este notebook en modo **offline (1 vez al d?a)**, no en loop r?pido.


**Qué hace `agent_04_refresh_ping_range_master.ipynb`**

Ruta:

- C:\TSIS_Data\v1\backtest_SmallCaps\notebooks\00_data_certification\agent_04_refresh_ping_range_master.ipynb

Función real:

1. Carga un universo de tickers (UNIVERSE_PARQUET).
2. Para cada ticker consulta Polygon en /v2/aggs/ticker/{ticker}/range/1/day/{PING_START}/{PING_END}:
    - sort=asc, limit=1 para first_day
    - sort=desc, limit=1 para last_day
3. Genera un master con:
    - ticker, has_data, first_day, last_day
4. Guarda:
    - ping_range_master.parquet
    - ping_range_master.csv

Es el agente que “pregunta a Polygon” el rango operativo real por ticker.


In [ ]:
from pathlib import Path
import os
import time
import json
import requests
import pandas as pd

# --- Config ---
POLYGON_API_KEY = os.getenv("POLYGON_API_KEY", "")  # recomendado usar variable de entorno

# Universo fuente (ajusta al parquet real de tu universo)
UNIVERSE_PARQUET = Path(r"C:\TSIS_Data\v1\backtest_SmallCaps\data\processed\universe\hybrid_enriched_2025-11-01.parquet")

# Ventana para ping (operativa)
PING_START = "2004-01-01"
PING_END = "2026-12-31"

OUT_DIR = Path(r"C:\TSIS_Data\v1\backtest_SmallCaps\data\reference")
OUT_DIR.mkdir(parents=True, exist_ok=True)
PING_MASTER_PARQUET = OUT_DIR / "ping_range_master.parquet"
PING_MASTER_CSV = OUT_DIR / "ping_range_master.csv"

# Control de velocidad
SLEEP_PER_TICKER_SEC = 0.12
TIMEOUT_SEC = 30

print("UNIVERSE_PARQUET:", UNIVERSE_PARQUET)
print("PING_MASTER_PARQUET:", PING_MASTER_PARQUET)
print("POLYGON_API_KEY loaded:", bool(POLYGON_API_KEY))


In [ ]:
# --- Cargar universo de tickers ---
if not UNIVERSE_PARQUET.exists():
    raise FileNotFoundError(f"No existe universo: {UNIVERSE_PARQUET}")

u = pd.read_parquet(UNIVERSE_PARQUET)

# detectar columna ticker
ticker_col = None
for c in ["ticker", "symbol", "Ticker", "SYMBOL"]:
    if c in u.columns:
        ticker_col = c
        break

if ticker_col is None:
    raise RuntimeError("No se encontr? columna ticker/symbol en el universo.")

tickers = (
    u[ticker_col]
    .astype(str)
    .str.strip()
    .replace("", pd.NA)
    .dropna()
    .drop_duplicates()
    .sort_values()
    .tolist()
)

print("ticker_col:", ticker_col)
print("tickers_total:", len(tickers))
print("sample:", tickers[:10])


In [ ]:
# --- Ping helpers ---
BASE = "https://api.polygon.io"


def _ping_side(ticker: str, sort: str):
    url = f"{BASE}/v2/aggs/ticker/{ticker}/range/1/day/{PING_START}/{PING_END}"
    params = {
        "adjusted": "true",
        "sort": sort,     # asc -> first_day, desc -> last_day
        "limit": 1,
        "apiKey": POLYGON_API_KEY,
    }
    r = requests.get(url, params=params, timeout=TIMEOUT_SEC)
    if r.status_code != 200:
        return {"ok": False, "status_code": r.status_code, "data": None, "error": r.text[:400]}

    js = r.json()
    results = js.get("results", [])
    if not results:
        return {"ok": True, "status_code": 200, "data": None, "error": None}

    t_ms = results[0].get("t")
    dt = pd.to_datetime(t_ms, unit="ms", utc=True).date() if t_ms is not None else None
    return {"ok": True, "status_code": 200, "data": str(dt) if dt is not None else None, "error": None}


def ping_ticker_range(ticker: str):
    asc = _ping_side(ticker, "asc")
    desc = _ping_side(ticker, "desc")

    has_data = bool(asc.get("data") and desc.get("data"))
    return {
        "ticker": ticker,
        "has_data": has_data,
        "first_day": asc.get("data"),
        "last_day": desc.get("data"),
        "asc_status": asc.get("status_code"),
        "desc_status": desc.get("status_code"),
        "asc_error": asc.get("error"),
        "desc_error": desc.get("error"),
    }


In [ ]:
# --- Ejecutar ping masivo ---
if not POLYGON_API_KEY:
    raise RuntimeError("Falta POLYGON_API_KEY. Exporta la variable de entorno antes de ejecutar.")

rows = []
for i, tk in enumerate(tickers, start=1):
    out = ping_ticker_range(tk)
    rows.append(out)

    if i % 100 == 0:
        print(f"processed={i}/{len(tickers)}")

    time.sleep(SLEEP_PER_TICKER_SEC)

ping_df = pd.DataFrame(rows)
ping_df["updated_at_utc"] = pd.Timestamp.utcnow().isoformat()

print("rows:", len(ping_df))
print("has_data true:", int(ping_df["has_data"].fillna(False).sum()))
print("has_data false:", int((~ping_df["has_data"].fillna(False)).sum()))

display(ping_df.head(10))


In [ ]:
# --- Persistir master ---
ping_df.to_parquet(PING_MASTER_PARQUET, index=False)
ping_df.to_csv(PING_MASTER_CSV, index=False, encoding="utf-8")

print("saved:", PING_MASTER_PARQUET)
print("saved:", PING_MASTER_CSV)
